In [1]:
# Clone the project's Megha branch
!git clone -b megha-work https://github.com/meghagangal362015/cmpe257-early-reading-prediction.git

%cd cmpe257-early-reading-prediction

Cloning into 'cmpe257-early-reading-prediction'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 61 (delta 14), reused 45 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 927.78 KiB | 10.42 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/cmpe257-early-reading-prediction


In [2]:
import pandas as pd
import numpy as np

DATA_PATH = "data/processed/ecri_combined.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Dataset shape: (5679, 46)
Columns:
['StuID', 'Gender', 'Tier', 'SchlID', 'TeachID', 'LEP', 'SpEd', 'Tx', 'Cohort', 'Age1b', 'Tier2_N', 'grp_rate', 'rcmistot', 'gnrl_fid', 'TKPctCrct', 'Tier2MinRd', 'TYearsTeaching', 'NWFcls', 'NWFwrc', 'ORFwc', 'SAwrS', 'SAsrS', 'SAtoS', 'RMwidRS', 'RMwdaRS', 'RMwidRS_target', 'RMwdaRS_target', 'Schl_FreeLunch', 'Schl_AmericanIndian', 'Schl_Asian', 'Schl_Hispanic', 'Schl_AfricanAmerican', 'Schl_White', 'Schl_NativeHawaiian', 'Schl_MoreRaces', 'NWFcls_mid', 'NWFwrc_mid', 'ORFwc_mid', 'SAtoS_spring', 'RMwidRS_spring', 'RMwdaRS_spring', 'SAtoS_gain', 'RMwidRS_gain', 'RMwdaRS_gain', 'SAtoS_target', 'split']


In [3]:
# Check the two paper targets

targets = ["RMwidRS_target", "RMwdaRS_target"]

for target in targets:
    print(f"\n{target}")
    print("Labeled students:", df[target].notna().sum())
    print("Missing labels:", df[target].isna().sum())
    print("Class counts:")
    print(df[target].value_counts(dropna=True).sort_index())


RMwidRS_target
Labeled students: 946
Missing labels: 4733
Class counts:
RMwidRS_target
0.0    447
1.0    499
Name: count, dtype: int64

RMwdaRS_target
Labeled students: 946
Missing labels: 4733
Class counts:
RMwdaRS_target
0.0    401
1.0    545
Name: count, dtype: int64


In [4]:
# Features used in the paper

paper_features = [
    # Initial reading assessment features
    "NWFcls",
    "NWFwrc",
    "ORFwc",
    "SAwrS",
    "SAsrS",
    "SAtoS",
    "RMwidRS",
    "RMwdaRS",

    # Student and classroom features
    "Gender",
    "Tx",
    "Age1b",
    "Tier2_N",
    "TKPctCrct",
    "grp_rate",
    "rcmistot",
    "gnrl_fid",
]

print("Number of paper features:", len(paper_features))

missing_features = [
    feature for feature in paper_features
    if feature not in df.columns
]

print("Missing feature names:", missing_features)

Number of paper features: 16
Missing feature names: []


In [5]:
# Keep only students with all 16 features and both paper targets available

complete_df = df.dropna(
    subset=paper_features + targets
).copy()

print("Complete dataset shape:", complete_df.shape)
print("Number of schools:", complete_df["SchlID"].nunique())
print("Missing values in selected data:", complete_df[paper_features + targets].isna().sum().sum())
print("\nTarget counts:")
print(complete_df[targets].value_counts())

Complete dataset shape: (706, 46)
Number of schools: 42
Missing values in selected data: 0

Target counts:
RMwidRS_target  RMwdaRS_target
1.0             1.0               302
0.0             0.0               209
                1.0               114
1.0             0.0                81
Name: count, dtype: int64


In [6]:
# Separate inputs, targets, and school groups

X = complete_df[paper_features].copy()

y_word_identification = complete_df["RMwidRS_target"].astype(int)
y_word_attack = complete_df["RMwdaRS_target"].astype(int)

groups = complete_df["SchlID"]

print("Feature matrix:", X.shape)
print("Word Identification target:", y_word_identification.shape)
print("Word Attack target:", y_word_attack.shape)
print("Number of school groups:", groups.nunique())
print("\nFeature data types:")
print(X.dtypes)

Feature matrix: (706, 16)
Word Identification target: (706,)
Word Attack target: (706,)
Number of school groups: 42

Feature data types:
NWFcls       float64
NWFwrc       float64
ORFwc        float64
SAwrS        float64
SAsrS        float64
SAtoS        float64
RMwidRS      float64
RMwdaRS      float64
Gender       float64
Tx             int64
Age1b        float64
Tier2_N      float64
TKPctCrct    float64
grp_rate     float64
rcmistot     float64
gnrl_fid     float64
dtype: object


In [7]:
# Identify feature types

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

Categorical features: []
Numeric features: ['NWFcls', 'NWFwrc', 'ORFwc', 'SAwrS', 'SAsrS', 'SAtoS', 'RMwidRS', 'RMwdaRS', 'Gender', 'Tx', 'Age1b', 'Tier2_N', 'TKPctCrct', 'grp_rate', 'rcmistot', 'gnrl_fid']


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GroupKFold

# Preprocess numeric and categorical columns separately
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Five folds, grouped by school
group_kfold = GroupKFold(n_splits=5)

print("Preprocessing pipeline created.")
print("Cross-validation:", group_kfold)

Preprocessing pipeline created.
Cross-validation: GroupKFold(n_splits=5, random_state=None, shuffle=False)


In [9]:
# Verify that each validation fold contains different schools
# from its training fold

for fold, (train_index, validation_index) in enumerate(
    group_kfold.split(X, y_word_identification, groups=groups),
    start=1
):
    train_schools = set(groups.iloc[train_index])
    validation_schools = set(groups.iloc[validation_index])

    overlap = train_schools.intersection(validation_schools)

    print(
        f"Fold {fold}: "
        f"training students = {len(train_index)}, "
        f"validation students = {len(validation_index)}, "
        f"training schools = {len(train_schools)}, "
        f"validation schools = {len(validation_schools)}, "
        f"overlapping schools = {len(overlap)}"
    )

Fold 1: training students = 564, validation students = 142, training schools = 34, validation schools = 8, overlapping schools = 0
Fold 2: training students = 564, validation students = 142, training schools = 33, validation schools = 9, overlapping schools = 0
Fold 3: training students = 566, validation students = 140, training schools = 34, validation schools = 8, overlapping schools = 0
Fold 4: training students = 566, validation students = 140, training schools = 34, validation schools = 8, overlapping schools = 0
Fold 5: training students = 564, validation students = 142, training schools = 33, validation schools = 9, overlapping schools = 0


In [10]:
# Save the paper-aligned complete dataset

paper_complete_df = complete_df[
    ["SchlID"] + paper_features + targets
].copy()

output_path = "data/processed/paper_complete_706.csv"
paper_complete_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Saved shape:", paper_complete_df.shape)

Saved: data/processed/paper_complete_706.csv
Saved shape: (706, 19)


In [11]:
# Verify the saved paper-aligned dataset

check_df = pd.read_csv("data/processed/paper_complete_706.csv")

print("Shape:", check_df.shape)
print("Missing values:", check_df.isna().sum().sum())
print("Schools:", check_df["SchlID"].nunique())

print("\nWord Identification target counts:")
print(check_df["RMwidRS_target"].value_counts().sort_index())

print("\nWord Attack target counts:")
print(check_df["RMwdaRS_target"].value_counts().sort_index())

Shape: (706, 19)
Missing values: 0
Schools: 42

Word Identification target counts:
RMwidRS_target
0.0    323
1.0    383
Name: count, dtype: int64

Word Attack target counts:
RMwdaRS_target
0.0    290
1.0    416
Name: count, dtype: int64
